In [3]:
!pip install -q kagglehub albumentations opencv-python scikit-learn torchvision torch torchaudio


In [4]:
import os
import random
from glob import glob
from typing import List, Tuple

import numpy as np
from PIL import Image

import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm

# ---- CONFIG ----
CLASS_NAMES = ["real", "fake"]
NUM_CLASSES = len(CLASS_NAMES)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

VAL_SIZE = 0.2
RANDOM_STATE = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [5]:
def download_dataset() -> str:
    path = kagglehub.dataset_download("yuvrajpaikhot/extracted-deepfake-frames")
    print("Dataset path:", path)
    print("Top-level contents:", os.listdir(path))
    return path

def collect_paths(root_folder: str) -> Tuple[List[str], List[int]]:
    """
    root_folder has train/, val/, test/ with subfolders for real/fake.
    real_* -> label 0, fake_* -> label 1
    """
    image_paths = []
    labels = []

    for split in ["train", "val"]:
        split_dir = os.path.join(root_folder, split)
        if not os.path.isdir(split_dir):
            continue

        for folder in os.listdir(split_dir):
            folder_path = os.path.join(split_dir, folder)
            if not os.path.isdir(folder_path):
                continue

            name = folder.lower()
            if name.startswith("real"):
                label = 0
            elif name.startswith("fake"):
                label = 1
            else:
                continue

            files = glob(os.path.join(folder_path, "*.jpg")) + glob(os.path.join(folder_path, "*.png"))
            image_paths.extend(files)
            labels.extend([label] * len(files))

    print("Total images:", len(image_paths))
    print("Real images:", sum(1 for l in labels if l == 0))
    print("Fake images:", sum(1 for l in labels if l == 1))
    return image_paths, labels

dataset_path = download_dataset()
all_paths, all_labels = collect_paths(dataset_path)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=all_labels,
)

print("Train size:", len(train_paths), "Val size:", len(val_paths))


Dataset path: /kaggle/input/datasets/yuvrajpaikhot/extracted-deepfake-frames
Top-level contents: ['val', 'splits_info.json', 'test', 'train']
Total images: 72875
Real images: 7700
Fake images: 65175
Train size: 58300 Val size: 14575


In [6]:
def get_train_val_transforms():
    train_transform = A.Compose(
        [
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.GaussianBlur(blur_limit=3, p=0.2),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ]
    )

    val_transform = A.Compose(
        [
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ]
    )

    return train_transform, val_transform

class DeepfakeDataset(Dataset):
    def __init__(self, paths: List[str], labels: List[int], transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = self.labels[idx]

        img = np.array(Image.open(img_path).convert("RGB"))

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        return img, label

train_tf, val_tf = get_train_val_transforms()

train_ds = DeepfakeDataset(train_paths, train_labels, transform=train_tf)
val_ds = DeepfakeDataset(val_paths, val_labels, transform=val_tf)

len(train_ds), len(val_ds)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


(58300, 14575)

In [7]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Batches – train:", len(train_loader), "val:", len(val_loader))


Batches – train: 3644 val: 911


In [8]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    ResNet-50 WITH ImageNet pretrained weights (needs internet once).
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

# Recreate model after changing definition
model = build_model().to(DEVICE)
print("Model ready on:", DEVICE)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s] 


Model ready on: cuda


In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = running_loss / total
    acc = 100.0 * correct / total
    return avg_loss, acc, all_labels, all_preds


In [11]:

def main_train():
    model = build_model(NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    best_val_acc = 0.0
    os.makedirs("models", exist_ok=True)
    best_path = "models/deepfake_resnet50_clean.pth"

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, y_true, y_pred = validate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}%")
        print(f"Val   loss: {val_loss:.4f} | Val   acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print(f"✅ New best model saved with val_acc={best_val_acc:.2f}%")

    print("\nTraining finished.")
    print(f"Best val acc: {best_val_acc:.2f}%")

    print("\nClassification report (last epoch):")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

# Call this in a separate cell so it's obvious where training starts
main_train()



Epoch 1/10


Train:   0%|          | 0/3644 [00:00<?, ?it/s]

Val:   0%|          | 0/911 [00:00<?, ?it/s]

Train loss: 0.3290 | Train acc: 92.73%
Val   loss: 0.3062 | Val   acc: 93.81%
✅ New best model saved with val_acc=93.81%

Epoch 2/10


Train:   0%|          | 0/3644 [00:00<?, ?it/s]

...

Val:   0%|          | 0/911 [00:00<?, ?it/s]

Train loss: 0.2549 | Train acc: 96.99%
Val   loss: 0.2688 | Val   acc: 96.32%

Epoch 10/10


Train:   0%|          | 0/3644 [00:00<?, ?it/s]

Val:   0%|          | 0/911 [00:00<?, ?it/s]

Train loss: 0.2507 | Train acc: 97.22%
Val   loss: 0.2655 | Val   acc: 96.45%
✅ New best model saved with val_acc=96.45%

Training finished.
Best val acc: 96.45%

Classification report (last epoch):
              precision    recall  f1-score   support

        real       0.93      0.71      0.81      1540
        fake       0.97      0.99      0.98     13035

    accuracy                           0.96     14575
   macro avg       0.95      0.85      0.89     14575
weighted avg       0.96      0.96      0.96     14575

Confusion matrix:
[[ 1099   441]
 [   77 12958]]


In [12]:
# Reload best model
best_model = build_model(NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(torch.load("models/deepfake_resnet50_clean.pth", map_location=DEVICE))
best_model.eval()

# Random val image
idx = random.randint(0, len(val_paths) - 1)
test_path = val_paths[idx]
print("Test image path:", test_path)

_, val_tf = get_train_val_transforms()
img = np.array(Image.open(test_path).convert("RGB"))
img_t = val_tf(image=img)["image"].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = best_model(img_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

pred_idx = int(np.argmax(probs))
print("Predicted:", CLASS_NAMES[pred_idx])
print("Probabilities (real, fake):", probs)


Test image path: /kaggle/input/datasets/yuvrajpaikhot/extracted-deepfake-frames/train/fake_train_00351/frame_000.jpg
Predicted: fake
Probabilities (real, fake): [0.04837739 0.95162255]


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
